# Week 1 · Notebook 2 — Indicators: turning prices into signals

A raw price tells you almost nothing. Is 18.50 high or low? Rising or calm? An
**indicator** is a small calculation over recent prices that answers one such
question with a number. Today you build two, and put them on the chart tool you made
yesterday.

Two functions you build:
1. `sma` — the simple moving average (the trend).
2. `rsi` — the relative strength index (momentum: overbought vs oversold).

## 1. Function — `sma` (simple moving average)

The average of the last `window` prices, recomputed each day. It smooths daily
noise so a trend becomes visible. Because it needs `window` days of history before
it can produce a value, the first `window-1` entries are `NaN`.

**In:** `prices`, `window`. **Out:** an array the same length as `prices`, `NaN`
for the first `window-1` entries.
**Hint:** for each `i`, average `prices[i-window+1 : i+1]`.
**Done when:** the check passes.

In [ ]:
import sys, os
while not os.path.isdir('src') and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir('..')
sys.path.insert(0, 'src')

import numpy as np
import matplotlib.pyplot as plt
from tradinglab.data_feed import DataFeed

feed = DataFeed.from_dir('data/egx'); 
price = feed.close[:, 0]

def sma(prices, window):
    prices = np.asarray(prices, dtype=float)
    out = np.full_like(prices, np.nan)
    # ---8<--- solution
    for i in range(window-1, len(prices)):
        out[i] = prices[i-window+1:i+1].mean()
    # ---8<--- end
    return out

t = sma(np.array([1.,2,3,4,5]), 3)
assert np.isnan(t[:2]).all() and t[2]==2.0 and t[4]==4.0, 'not right yet'
print('sma correct ✓')

## 2. Function — `rsi` (relative strength index)

RSI measures how hard price has been pushed up vs down recently, on a 0–100 scale.
Above ~70 is often called "overbought", below ~30 "oversold". The recipe: average
the up-moves and the down-moves over a window, form `rs = avg_gain / avg_loss`, then
`100 - 100/(1+rs)`.

**In:** `prices`, `window` (default 14). **Out:** array in [0, 100], `NaN` early.
**Hint:** `delta = np.diff(prices)`; gains are positive deltas, losses the absolute
negative ones.
**Done when:** values stay within [0, 100].

In [ ]:
def rsi(prices, window=14):
    prices = np.asarray(prices, dtype=float)
    out = np.full_like(prices, np.nan)
    delta = np.diff(prices)
    gains = np.where(delta > 0, delta, 0.0)
    losses = np.where(delta < 0, -delta, 0.0)
    # ---8<--- solution
    for i in range(window, len(prices)):
        ag = gains[i-window:i].mean(); al = losses[i-window:i].mean()
        out[i] = 100.0 if al == 0 else 100.0 - 100.0/(1.0 + ag/al)
    # ---8<--- end
    return out

r = rsi(price, 14); valid = r[~np.isnan(r)]
assert (valid >= 0).all() and (valid <= 100).all(), 'RSI must be in [0,100]'
print('rsi correct ✓  (recent RSI:', round(np.nanmean(r[-20:]),1), ')')

## 3. See them on your chart
Reuse the `plot_price` tool you built yesterday. Indicators only mean something when
you can see them against price.

In [ ]:
from tradinglab.charting import plot_price   # your graduated tool
ax = plot_price(feed.dates, price,
                overlays={'SMA20': sma(price, 20), 'SMA50': sma(price, 50)},
                title=feed.symbols[0] + ' with moving averages')
plt.show()
print('Notice how the SMAs lag price and smooth the noise — that lag is the trade-off.')


In [ ]:
# --- RSI belongs in its own panel — different scale than price ---
r = rsi(price, 14)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7), sharex=True,
                                gridspec_kw={'height_ratios': [2, 1]})

ax1.plot(feed.dates, price, linewidth=1.2)
ax1.set_title(feed.symbols[0] + ' — price')
ax1.grid(alpha=0.3)

ax2.plot(feed.dates, r, color='purple', linewidth=1.0)
ax2.axhline(70, color='red', linestyle='--', linewidth=0.8, label='overbought (70)')
ax2.axhline(30, color='green', linestyle='--', linewidth=0.8, label='oversold (30)')
ax2.set_ylim(0, 100)
ax2.set_title('RSI(14)')
ax2.legend(loc='upper left', fontsize=8)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
import sys, os

while not os.path.isdir('src') and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir('..')

sys.path.insert(0, 'src')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from tradinglab.data_feed import DataFeed


# Load data
feed = DataFeed.from_dir('data/egx')

print("Universe:", feed.symbols)
print("Days:", feed.n_days)


# Select stock
symbol = feed.symbols[0]
prices = feed.close[:, 0]

print("Trading:", symbol)


# Moving average windows
fast_window = 20
slow_window = 50


# Calculate moving averages
fast_ma = np.convolve(
    prices,
    np.ones(fast_window) / fast_window,
    mode='valid'
)

slow_ma = np.convolve(
    prices,
    np.ones(slow_window) / slow_window,
    mode='valid'
)


# Align fast MA with slow MA
fast_ma_aligned = fast_ma[slow_window - fast_window:]


# Generate signals
signals = np.zeros(len(slow_ma))


for t in range(1, len(slow_ma)):

    # BUY: fast MA crosses above slow MA
    if (
        fast_ma_aligned[t - 1] <= slow_ma[t - 1]
        and fast_ma_aligned[t] > slow_ma[t]
    ):
        signals[t] = 1

    # SELL: fast MA crosses below slow MA
    elif (
        fast_ma_aligned[t - 1] >= slow_ma[t - 1]
        and fast_ma_aligned[t] < slow_ma[t]
    ):
        signals[t] = -1


# Get signal locations
buy_days = np.where(signals == 1)[0]
sell_days = np.where(signals == -1)[0]


print("BUY signals :", len(buy_days))
print("SELL signals:", len(sell_days))


# Plot
plot_prices = prices[-len(slow_ma):]
plot_dates = feed.dates[-len(slow_ma):]


plt.figure(figsize=(14, 7))


# Closing price
plt.plot(
    plot_dates,
    plot_prices,
    label="Close"
)


# Fast Moving Average
plt.plot(
    plot_dates,
    fast_ma_aligned,
    label="Fast MA (20)"
)


# Slow Moving Average
plt.plot(
    plot_dates,
    slow_ma,
    label="Slow MA (50)"
)


# BUY signals
plt.scatter(
    plot_dates[buy_days],
    plot_prices[buy_days],
    marker="^",
    s=100,
    label="BUY"
)


# SELL signals
plt.scatter(
    plot_dates[sell_days],
    plot_prices[sell_days],
    marker="v",
    s=100,
    label="SELL"
)


# Chart settings
plt.title(f"Moving Average Crossover - {symbol}")
plt.xlabel("Year")
plt.ylabel("Price")

plt.legend()
plt.grid()


# Show years on X-axis
plt.gca().xaxis.set_major_locator(mdates.YearLocator())
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y'))


plt.show()

In [ ]:
import sys, os

while not os.path.isdir('src') and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir('..')

sys.path.insert(0, 'src')

import numpy as np
import pandas as pd

from tradinglab.data_feed import DataFeed


# ============================================================
# 1. Load data
# ============================================================

feed = DataFeed.from_dir('data/egx')

print("Universe:", feed.symbols)
print("Number of stocks:", len(feed.symbols))
print("Days:", feed.n_days)


# ============================================================
# 2. Parameters
# ============================================================

initial_capital = 1000.0

fast_window = 20
slow_window = 50


# ============================================================
# 3. SMA
# ============================================================

def sma(prices, window):

    return np.convolve(
        prices,
        np.ones(window) / window,
        mode='valid'
    )


# ============================================================
# 4. Maximum Drawdown
# ============================================================

def calculate_max_drawdown(equity):

    running_max = np.maximum.accumulate(equity)

    drawdown = (
        equity - running_max
    ) / running_max

    return drawdown.min()


# ============================================================
# 5. Backtest one stock
# ============================================================

def backtest_stock(prices):

    # -----------------------------
    # Moving averages
    # -----------------------------

    fast_ma = sma(prices, fast_window)

    slow_ma = sma(prices, slow_window)

    # Align Fast MA with Slow MA
    fast_ma = fast_ma[
        slow_window - fast_window:
    ]

    # Align prices
    prices = prices[-len(slow_ma):]


    # -----------------------------
    # Generate signals
    # -----------------------------

    signals = np.zeros(len(slow_ma))

    for t in range(1, len(slow_ma)):

        # BUY
        if (
            fast_ma[t - 1] <= slow_ma[t - 1]
            and fast_ma[t] > slow_ma[t]
        ):
            signals[t] = 1

        # SELL
        elif (
            fast_ma[t - 1] >= slow_ma[t - 1]
            and fast_ma[t] < slow_ma[t]
        ):
            signals[t] = -1


    # -----------------------------
    # Position
    # -----------------------------

    position = np.zeros(len(signals))

    current_position = 0

    for t in range(len(signals)):

        if signals[t] == 1:
            current_position = 1

        elif signals[t] == -1:
            current_position = 0

        position[t] = current_position


    # -----------------------------
    # Daily returns
    # -----------------------------

    returns = np.zeros(len(prices))

    returns[1:] = (
        prices[1:] / prices[:-1] - 1
    )


    # -----------------------------
    # Strategy returns
    # -----------------------------

    strategy_returns = position * returns


    # -----------------------------
    # Equity curve
    # -----------------------------

    equity = np.zeros(len(strategy_returns))

    equity[0] = initial_capital

    for t in range(1, len(strategy_returns)):

        equity[t] = (
            equity[t - 1]
            * (1 + strategy_returns[t])
        )


    # -----------------------------
    # Results
    # -----------------------------

    final_value = equity[-1]

    final_profit = final_value - initial_capital

    max_dd = calculate_max_drawdown(equity)

    return final_value, final_profit, max_dd


# ============================================================
# 6. Apply to every stock
# ============================================================

results = []

for i, symbol in enumerate(feed.symbols):

    prices = feed.close[:, i].astype(float)

    # Skip stocks with missing data
    if np.isnan(prices).any():

        print(f"Skipping {symbol}: missing data")
        continue

    try:

        final_value, profit, max_dd = backtest_stock(prices)

        results.append({
            "Symbol": symbol,
            "Initial Capital": initial_capital,
            "Final Value": final_value,
            "Final Profit": profit,
            "Max Drawdown": max_dd
        })

    except Exception as e:

        print(f"Error with {symbol}: {e}")


# ============================================================
# 7. Create results table
# ============================================================

results_df = pd.DataFrame(results)


# ============================================================
# 8. Sort by Final Profit
# ============================================================

results_df = results_df.sort_values(
    "Final Profit",
    ascending=False
)


# ============================================================
# 9. Display results
# ============================================================

print("\n")
print("=" * 90)
print("MOVING AVERAGE CROSSOVER - WHOLE MARKET")
print("=" * 90)

print(
    results_df.to_string(
        index=False,
        formatters={
            "Initial Capital": "{:,.2f}".format,
            "Final Value": "{:,.2f}".format,
            "Final Profit": "{:,.2f}".format,
            "Max Drawdown": "{:.2%}".format
        }
    )
)


# ============================================================
# 10. Summary
# ============================================================

print("\n")
print("=" * 90)
print("SUMMARY")
print("=" * 90)

print(
    f"Each stock starts with: {initial_capital:,.2f} EGP"
)

print(
    f"Number of stocks tested: {len(results_df)}"
)

print(
    f"Average final value: "
    f"{results_df['Final Value'].mean():,.2f} EGP"
)

print(
    f"Average final profit: "
    f"{results_df['Final Profit'].mean():,.2f} EGP"
)

print(
    f"Best stock: "
    f"{results_df.iloc[0]['Symbol']} "
    f"({results_df.iloc[0]['Final Profit']:,.2f} EGP profit)"
)

print(
    f"Worst stock: "
    f"{results_df.iloc[-1]['Symbol']} "
    f"({results_df.iloc[-1]['Final Profit']:,.2f} EGP profit)"
)

## 4. Graduate and reflect
You now have `sma` and `rsi`. Move them into `src/tradinglab/indicators.py` and run
`uv run pytest week1/tests/`.

These aren't just charts — tomorrow they become **signals**. When a short SMA rises
above a long one, that's a trend you can trade. That's the strategy you build next.